In [ ]:
import os
import json
import logging
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from pprint import pprint
from scipy.spatial.distance import euclidean
from transformers import AutoTokenizer
from sklearn.metrics import pairwise_distances
import spacy


# SpaCy

In [ ]:
section_types = ['TITLE', 'ABSTRACT', 'INTRO', 'FIG', 'METHODS', 'RESULTS']
directory = os.getcwd()
data_rows = []
tokenize = spacy.load("en_core_web_sm")
# Iterate over all JSON files in the directory

for filename in os.listdir(directory):
    if filename.endswith(".json"):
        logging.info(f"Processing file {filename}")
        try:
            with open(os.path.join(directory, filename), 'r') as file:
                data = json.load(file)
                base_filename = os.path.splitext(filename)[0]
                
                # Extract passages for each section type
                for section_type in section_types:
                    passages = data[0].get('documents', [{}])[0].get('passages', [])
                    
                    for passage in passages:
                        # Check if the passage's section type matches and skip if type is "title_1"
                        if (passage.get('infons', {}).get('section_type') == section_type and 
                                passage.get('infons', {}).get('type') != "title_1"):
                            # Process the passage as needed
                            logging.info(f"Processing passage: {passage}")
                            
                            text = passage.get('text', '')
                            # Process the text with SpaCy
                            doc = tokenize(text)
                            
                            # Append each sentence as a new row in data_rows
                            for sentence in doc.sents:
                                data_rows.append({
                                    "filename": base_filename,
                                    "section_type": section_type,
                                    "sentence": sentence.text,  # Store the individual sentence
                                    "type": passage.get('infons', {}).get('type'),  # Get the type from the passage
                                    "text": text  # Keep original text
                                })
                        else:
                            logging.info(f"Skipping passage: {passage}")
        
        except json.JSONDecodeError as e:
            logging.error(f"Error parsing JSON file {filename}: {e}")
        except KeyError as e:
            logging.error(f"Error extracting text from JSON file {filename}: {e}")
        except Exception as e:
            logging.error(f"An unexpected error occurred while processing {filename}: {e}")

# Create a DataFrame from extracted rows
df = pd.DataFrame(data_rows)
logging.info(f"Extracted {len(df)} passages")

# Generate embeddings

In [ ]:
model = SentenceTransformer('microsoft/biogpt')

# Generate embeddings for the 'text' column
df['embeddings'] = df['sentence'].apply(lambda x: model.encode(x, show_progress_bar=False))

logging.info("Generated embeddings for all sentences.")

# compute cosine similarity between different sections

In [ ]:
embeddings_matrix = np.vstack(df['embeddings'].values) #stack (embeddings) vertically to create a matrix suitable for further computations, such as calculating cosine similarities.

similarities_matrix = cosine_similarity(embeddings_matrix)

# Create a dictionary to store the similarities
similarities = {}

# Using a nested loop..... iterate over section types to fill the distances dictionary
for i, section1 in enumerate(section_types):
    for j, section2 in enumerate(section_types):
        if i >= j:  # Avoid redundant comparisons
            continue

        # Store the similarity in the dictionary as a tuple (section1, section2)
        similarities[(section1, section2)] = similarities_matrix[i, j]

# Convert similarities to a DataFrame for better readability
similarity_df = pd.DataFrame.from_dict(similarities, orient='index', columns=['similarity'])

In [ ]:
distances_matrix = pairwise_distances(embeddings_matrix, metric='euclidean')

# Create a dictionary to store the distances
distances = {}

# Using a nested loop..... iterate over section types to fill the distances dictionary
for i, section1 in enumerate(section_types):
    for j, section2 in enumerate(section_types):
        if i >= j:  # Avoid redundant comparisons
            continue

        # Store the distance in the dictionary as a tuple (section1, section2)
        distances[(section1, section2)] = distances_matrix[i, j]

# Convert distances to a DataFrame for better readability
distance_df = pd.DataFrame.from_dict(distances, orient='index', columns=['distance'])